# Workshop 2: Retrieval Basics with RAG
## Theme Example: Sustainability Reports

In this notebook, we work with sustainability reports in PDF format.

Unlike plain text files, PDFs need a document loader to extract the text before we can process it for retrieval.

Our goal is to:
- load PDF documents
- split them into chunks
- create embeddings
- store them in a vector database
- retrieve relevant chunks for a query

## Working with PDF Documents

PDF files are common in real-world applications, but they are less straightforward to process than plain text.

To use them in a RAG pipeline, we first extract the text from each page. After that, we can split the content into chunks and build a retrieval system.

In [1]:
# Install required libraries (run once)
!pip install -q --upgrade langchain langchain-community langchain-text-splitters sentence-transformers faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 17.1 MB/s eta 0:00:00


In [3]:
#Library imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [4]:
#Downloading the PDF files
import requests

urls = [
    "https://www.theccc.org.uk/wp-content/uploads/2025/06/Progress-in-reducing-emissions-2025-report-to-Parliament.pdf",
    "https://assets.publishing.service.gov.uk/media/642d85fcfbe620000c17dd71/Net_Zero_Society_Report_2023.pdf"
]

file_names = []

for i, url in enumerate(urls):
    file_name = f"report_{i}.pdf"
    response = requests.get(url)

    with open(file_name, "wb") as f:
        f.write(response.content)

    file_names.append(file_name)

print("PDFs downloaded:", file_names)

PDFs downloaded: ['report_0.pdf', 'report_1.pdf']


In [5]:
#Loading the PDF files
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file_name in file_names:
    loader = PyPDFLoader(file_name)
    documents.extend(loader.load())

print(f"Total pages loaded: {len(documents)}")

Total pages loaded: 328


## Splitting PDF Content into Chunks

Each page from the PDF is treated as a document.

We now split these documents into smaller chunks to make retrieval more effective.

As before:
- smaller chunks → more precise retrieval  
- overlap → preserves context  

In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Number of chunks:", len(chunks))
print(chunks[0].page_content[:500])

Number of chunks: 1207
June 2025 
 
 
Progress in reducing emissions 
2025 report to Parliament


## Creating Embeddings and Building the Vector Store

Now we convert each chunk into an embedding, which is a numeric representation of its meaning.

We then store these embeddings in FAISS, so that we can quickly retrieve the most relevant chunks for a query.

In [15]:
# Create embeddings
embedding_model = HuggingFaceEmbeddings()

# Create vector store
db = FAISS.from_documents(chunks, embedding_model)

print("Vector store created successfully!")

/tmp/ipykernel_3579/2248466205.py:2: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding_model = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store created successfully!


## Retrieving Relevant Information

Now we can search the reports using a query.

The system will compare the query with the stored embeddings and return the most relevant chunks from the reports.

In [16]:
query = " energy waste"

results = db.similarity_search(query, k=3)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:\n")
    print(r.page_content[:500])


Result 1:

land available due to more livestock -focussed farming practices. These scenarios use 
synthetic kerosene in aviation, instead of biofuels,  although it is worth noting this is an 
unproven net zero technology. 
 
 
i The source categories represent energy consumed through end-use sector technologies. For example, 
‘heat’ is s ppl ied from distri t heating s stems  ‘solar’ refers to solar thermal  and ‘non -energ   se’ is 
energy consumed in industry as feedstocks (for example, natural gas use f

Result 2:

result. As an alternative, we present a disaggregation by sector of how efficiency changes 
(for example,  home insulation or more fuel -efficient vehicles) interact wi th changes in 
activity or outputs (such as distance travelled, or the quantity of goods manufactured) to 
produce a net change in energy use. This is shown in Figure 24. While it is only feasible 
to carry out this analysis for energy use, is important to note that total energy use and the 
costs of meet

## Try Your Own Queries

Now try your own queries related to sustainability.

For example:
- emissions reduction  
- climate policy  
- renewable energy  
- carbon targets  

Observe:
- Are the results relevant?
- Do different queries return different types of information?

## Summary

In this notebook, we:
- loaded PDF reports  
- split them into chunks  
- created embeddings  
- stored them in a vector database  
- retrieved relevant information using queries  

This demonstrates how retrieval works with real-world documents.

Next, we will connect retrieval with a language model to generate answers.